In [1]:
import os
import warnings
from pathlib import Path

import datasets as hfds
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from sklearn.preprocessing import scale
from tqdm import tqdm

In [2]:
style_path = os.environ["MPL_STYLE"]
print("matplotlib style:", style_path)
plt.style.use(style_path)

PLOTW, _ = plt.rcParams["figure.figsize"]
FORMAT = plt.rcParams["savefig.format"]
print(PLOTW, FORMAT)

matplotlib style: /ocean/projects/med220004p/clane2/ARFC/arfc-experiments/resources/clane.mplstyle
3.42 png


In [3]:
fc_colors = np.array(
    [
        [64, 80, 160],
        [64, 96, 176],
        [96, 192, 240],
        [144, 208, 224],
        [255, 255, 255],
        [240, 240, 96],
        [240, 208, 64],
        [224, 112, 64],
        [224, 64, 48],
    ],
    dtype=np.uint8,
)

FC_CMAP = LinearSegmentedColormap.from_list("fc", fc_colors / 255.0)

In [4]:
HCP_TR = 0.72

project_root = Path(os.environ["PROJECT_ROOT"])

visualize_parc_dir = project_root / "results/visualize_parcellated_hcp_timeseries"
visualize_parc_dir.mkdir(exist_ok=True)

In [5]:
ts_dataset = hfds.load_from_disk(
    project_root / "data/hcp_1200_rfmri_schaefer_timeseries"
)
ts_dataset.set_format("numpy")
print(ts_dataset)
print(ts_dataset.features)

Loading dataset from disk:   0%|          | 0/157 [00:00<?, ?it/s]

Dataset({
    features: ['sub', 'ses', 'run', 'timeseries_n200', 'timeseries_n400', 'timeseries_n800', 'timeseries_n1000'],
    num_rows: 3468
})
{'sub': Value(dtype='string', id=None), 'ses': Value(dtype='uint8', id=None), 'run': Value(dtype='uint8', id=None), 'timeseries_n200': Array2D(shape=(None, 200), dtype='float32', id=None), 'timeseries_n400': Array2D(shape=(None, 400), dtype='float32', id=None), 'timeseries_n800': Array2D(shape=(None, 800), dtype='float32', id=None), 'timeseries_n1000': Array2D(shape=(None, 1000), dtype='float32', id=None)}


In [6]:
sample_ts_data = ts_dataset.remove_columns("timeseries_n1000").select(range(0, 400, 2))

In [7]:
def plot_carpet_grid(nrow: int, ncol: int, parc_size: int = 400):
    ploth = 1.75
    stride = 2

    plotw = ploth * parc_size / (1200 / stride)
    f, axs = plt.subplots(
        nrow,
        ncol,
        figsize=(ncol * plotw, nrow * ploth),
        layout="constrained",
    )

    axs = axs.flatten()
    for ii in range(nrow * ncol):
        plt.sca(axs[ii])
        sample = sample_ts_data[ii]
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            X = scale(sample[f"timeseries_n{parc_size}"])

        X = X[::stride]

        plt.imshow(X, interpolation="nearest", vmin=-3, vmax=3, cmap=FC_CMAP)
        plt.xticks([])
        plt.yticks([])
        plt.title("{sub} s{ses} r{run}".format(**sample), fontsize="x-small", pad=4)

    return f

In [8]:
nrow = 10
ncol = 20
parc_size = 400

f = plot_carpet_grid(nrow, ncol, parc_size=parc_size)
f.savefig(
    visualize_parc_dir
    / f"hcp_1200_rfmri_preproc_carpet_plot_parc-{parc_size}_{nrow}x{ncol}.{FORMAT}"
)

# Don't show the figure inline
plt.close(f)

In [9]:
nrow = 4
ncol = 8
parc_sizes = [200, 400, 800]

for parc_size in tqdm(parc_sizes):
    f = plot_carpet_grid(nrow, ncol, parc_size=parc_size)
    f.savefig(
        visualize_parc_dir
        / f"hcp_1200_rfmri_preproc_carpet_plot_parc-{parc_size}_{nrow}x{ncol}.{FORMAT}"
    )
    plt.close(f)

  0%|                                                                       | 0/3 [00:00<?, ?it/s]

 33%|█████████████████████                                          | 1/3 [00:02<00:04,  2.11s/it]

 67%|██████████████████████████████████████████                     | 2/3 [00:05<00:02,  2.90s/it]

100%|███████████████████████████████████████████████████████████████| 3/3 [00:12<00:00,  4.72s/it]

100%|███████████████████████████████████████████████████████████████| 3/3 [00:12<00:00,  4.15s/it]

In [10]:
def plot_rms_timeseries_grid(nrow: int, ncol: int, parc_size: int = 400):
    stride = 2

    ploth = 0.5
    plotw = 2.0

    f, axs = plt.subplots(
        nrow,
        ncol,
        figsize=(ncol * plotw, nrow * ploth),
        layout="constrained",
        sharex=True,
        sharey=True,
    )

    axs = axs.flatten()
    for ii in range(nrow * ncol):
        plt.sca(axs[ii])
        sample = sample_ts_data[ii]
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            X = scale(sample[f"timeseries_n{parc_size}"])

        X = X[::stride]
        rms = np.sqrt(np.mean(X**2, axis=1))
        plt.plot(rms, "k-", lw=1.0)
        plt.ylim(0, 2)
        plt.xlim(-1, len(rms))
        plt.xticks([])
        plt.yticks([])
        plt.title("{sub} s{ses} r{run}".format(**sample), fontsize="x-small", pad=4)

    return f

In [11]:
nrow = 8
ncol = 4
parc_size = 400

f = plot_carpet_grid(nrow, ncol, parc_size=parc_size)
f.savefig(
    visualize_parc_dir
    / f"hcp_1200_rfmri_preproc_carpet_plot_parc-{parc_size}_{nrow}x{ncol}.{FORMAT}"
)
plt.close(f)

f = plot_rms_timeseries_grid(nrow, ncol, parc_size=parc_size)
f.savefig(
    visualize_parc_dir
    / f"hcp_1200_rfmri_preproc_rms_timeseries_parc-{parc_size}_{nrow}x{ncol}.{FORMAT}"
)
plt.close(f)